# AI-sfa Wan LoRA 학습 시험 — Kaggle T4에서 학습이 되는지

사용자 GPU가 당분간 1050 Ti뿐이라 학습을 Kaggle로 옮길 수 있는지 잰다. 학습 자료 48조각, 100스텝만(결과 LoRA는 안 씀).
GPU 0 = fp16 학습, GPU 1 = bf16 학습(PC와 같은 방식, T4에선 에뮬레이션)을 동시에 돌려 속도·손실·메모리를 PC(2.85초/스텝, 손실 0.16)와 비교한다.

**설정:**
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**
3. (선택) **Add-ons → Secrets**에 `RCLONE_GDRIVE_AUTH_B64`를 붙이면 결과가 Drive `AI-sfa/02_분석자료/WanLoRA_평가/`로 올라간다. 없으면 Output 탭에서 받는다.

**실행:** **Save Version → Save & Run All (Commit)**. 예상 40–70분(추정). 결과 `wan_train_test.zip` 안 `train_test/summary.json`을 Claude에게.

In [ ]:
# 1. (선택) Drive 연결 — 비밀값이 없으면 건너뛴다. 인증값은 출력하지 않는다
import os, json, base64, pathlib, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    raw = UserSecretsClient().get_secret('RCLONE_GDRIVE_AUTH_B64').strip()
    d = json.loads(base64.b64decode(raw + '=' * (-len(raw) % 4)))
    p = pathlib.Path.home() / '.config/rclone'; p.mkdir(parents=True, exist_ok=True)
    lines = ['[gdrive]', 'type = drive', 'scope = drive.readonly,drive.file']
    if d.get('client_id'): lines.append('client_id = ' + d['client_id'])
    if d.get('client_secret'): lines.append('client_secret = ' + d['client_secret'])
    lines.append('token = ' + d['token'])
    (p / 'rclone.conf').write_text('\n'.join(lines) + '\n'); os.chmod(p / 'rclone.conf', 0o600)
    del raw, d, lines
    subprocess.run('cd /tmp && curl -sSLo rc.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip && unzip -qo rc.zip && cp rclone-*/rclone /usr/local/bin/', shell=True, check=True)
    print('Drive 연결됨')
except Exception as e:
    print('Drive 연결 없이 진행(결과는 Output 탭):', type(e).__name__)

In [ ]:
# 2. 작업자 파일 쓰기 → 실행(받기·설치 → 48조각 캐시 → fp16·bf16 학습 100스텝 → 요약)
open('/tmp/kaggle_wan_train_test_worker.py', 'w', encoding='utf-8').write("#!/usr/bin/env python3\n\"\"\"Kaggle T4 x2: Wan2.1 1.3B LoRA 학습이 T4에서 되는지 시험 (2026-10-28 밤).\n\n사용자 GPU가 당분간 1050 Ti(4GB)뿐이라 학습을 Kaggle로 옮길 수 있는지 먼저 잰다. T4는 bf16 연산이 에뮬레이션(느림)이라 두 방식을 동시에 비교한다.\n  GPU 0: DiT fp16 파일 + --mixed_precision fp16 (musubi가 fp16 DiT면 fp16/no만 허용)\n  GPU 1: DiT bf16 파일 + --mixed_precision bf16 (PC와 같은 방식, 느릴 수 있음)\n나머지 설정은 PC 학습(pc_wan_lora_run.py)과 같다: LoRA 32/16, AdamW 2e-4, shift 3.0, gradient checkpointing, 시드 42. fp8_base는 뺐다(T4 16GB면 필요 없음).\n자료: Drive wanlora.zip(482조각) 중 48조각만(10개마다 1개) — 캐시 시간·스텝 속도·손실·메모리만 본다. 결과 LoRA는 쓰지 않는다.\nPC 기준(10-28, 5060, 482조각): 2.85 s/it, avr_loss 10스텝 0.182 / 20 0.169 / 40 0.164 / 60 0.159 / 100 0.161.\n도우미(fetch·MODELS·set_t5_dtype)는 kaggle_wan_eval_worker.py를 같은 브랜치에서 받아 쓴다.\n사용(노트북): python kaggle_wan_train_test_worker.py [--out /kaggle/working] [--steps 100]\n\"\"\"\nimport argparse\nimport glob\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport zipfile\n\nW = os.environ.get(\"WAN_DIR\", \"/tmp/wan\")\nRAW = \"https://raw.githubusercontent.com/twoapple08/AI-sfa/ccr-bccf15a6-d6ee46/tools\"\nWANLORA_ZIP = \"https://drive.usercontent.google.com/download?export=download&confirm=t&id=1YdosWijIlDx0vaTvIxos8rhlkY3CRH8m\"\nPC_REF = {10: 0.182, 20: 0.169, 40: 0.164, 60: 0.159, 100: 0.161}\nT0 = time.time()\n\n\ndef say(*a):\n    print(f\"[{(time.time() - T0) / 60:5.1f}분]\", *a, flush=True)\n\n\ndef run(cmd, cwd, log, env=None, timeout=None, show=(\"Error\", \"error\", \"Traceback\", \"steps:\")):\n    t0 = time.time()\n    last = 0\n    with open(log, \"a\", encoding=\"utf-8\") as fh:\n        p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors=\"replace\")\n        for line in p.stdout:\n            fh.write(line)\n            if any(s in line for s in show) and time.time() - last > 30:  # 진행 표시는 30초에 한 줄만\n                say(os.path.basename(log), line.rstrip()[-140:])\n                last = time.time()\n            if timeout and time.time() - t0 > timeout:\n                p.kill()\n                break\n        p.wait()\n    return p.returncode, time.time() - t0\n\n\nclass MemWatch(threading.Thread):\n    \"\"\"nvidia-smi로 GPU별 최대 메모리(MiB)를 잰다.\"\"\"\n    def __init__(self):\n        super().__init__(daemon=True)\n        self.peak, self.stop = {}, False\n\n    def run(self):\n        while not self.stop:\n            r = subprocess.run([\"nvidia-smi\", \"--query-gpu=index,memory.used\", \"--format=csv,noheader,nounits\"], capture_output=True, text=True)\n            for line in r.stdout.strip().splitlines():\n                i, m = (int(x) for x in line.split(\",\"))\n                self.peak[i] = max(self.peak.get(i, 0), m)\n            time.sleep(5)\n\n\ndef parse_steps(log):\n    t = open(log, encoding=\"utf-8\", errors=\"replace\").read()\n    d = {}\n    for m in re.finditer(r\"steps: +\\d+%\\|[^|]*\\| (\\d+)/(\\d+) \\[[^,]*, +([\\d.]+)(s/it|it/s), avr_loss=([\\d.na]+)\", t):\n        sit = float(m.group(3)) if m.group(4) == \"s/it\" else 1 / float(m.group(3))\n        d[int(m.group(1))] = (round(sit, 2), m.group(5))\n    return d, \"avr_loss=nan\" in t.lower()\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--out\", default=\"/kaggle/working\")\n    ap.add_argument(\"--steps\", type=int, default=100)\n    ap.add_argument(\"--clips\", type=int, default=48)\n    a = ap.parse_args()\n    os.makedirs(W, exist_ok=True)\n    logdir = os.path.join(a.out, \"train_test\")\n    os.makedirs(logdir, exist_ok=True)\n    subprocess.run([\"curl\", \"-sSL\", \"-o\", f\"{W}/kaggle_wan_eval_worker.py\", f\"{RAW}/kaggle_wan_eval_worker.py\"], check=True)\n    sys.path.insert(0, W)\n    import torch\n    import kaggle_wan_eval_worker as ev\n    ng = torch.cuda.device_count()\n    say(\"GPU\", ng, \"장\", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))\n    x, y = torch.randn(256, 1024, device=\"cuda\"), torch.randn(1024, 256, device=\"cuda\")\n    say(f\"bf16 곱셈 상대 오차 {float(((x.bfloat16() @ y.bfloat16()).float() - x @ y).abs().mean() / (x @ y).abs().mean()):.4f}\")\n\n    # 설치·받기(평가 노트북과 같은 musubi 커밋)\n    mt = os.path.join(W, \"musubi\")\n    if not os.path.exists(mt):\n        subprocess.run([\"curl\", \"-sSL\", \"-o\", f\"{W}/mt.zip\", f\"https://github.com/kohya-ss/musubi-tuner/archive/{ev.MT}.zip\"], check=True)\n        with zipfile.ZipFile(f\"{W}/mt.zip\") as z:\n            z.extractall(W)\n        os.rename(os.path.join(W, f\"musubi-tuner-{ev.MT}\"), mt)\n    say(\"musubi-tuner 설치\")\n    subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"-e\", mt], check=True, stdout=subprocess.DEVNULL)\n    ev.set_t5_dtype(mt, \"bfloat16\")\n    m = os.path.join(W, \"models\")\n    os.makedirs(m, exist_ok=True)\n    ths = [threading.Thread(target=ev.fetch, args=(f\"{ev.HF}/{ev.MODELS[k][0]}\", os.path.join(m, os.path.basename(ev.MODELS[k][0])), ev.MODELS[k][1]))\n           for k in (\"dit_fp16\", \"dit_bf16\", \"vae\", \"t5\")]\n    zp = os.path.join(W, \"wanlora.zip\")\n    ths.append(threading.Thread(target=lambda: os.path.exists(zp) or subprocess.run([\"curl\", \"-sSL\", \"--retry\", \"5\", \"-o\", zp, WANLORA_ZIP], check=True)))\n    say(\"모델 4개 + 학습 자료(wanlora.zip 98MB) 받는 중\")\n    for t in ths:\n        t.start()\n    for t in ths:\n        t.join()\n\n    # 자료 일부(10개마다 1개)\n    data = os.path.join(W, \"data\")\n    vid = os.path.join(data, \"video\")\n    if not os.path.exists(vid):\n        os.makedirs(vid)\n        with zipfile.ZipFile(zp) as z:\n            mp4 = sorted(n for n in z.namelist() if n.startswith(\"video/\") and n.endswith(\".mp4\"))\n            step = max(1, len(mp4) // a.clips)\n            for n in mp4[::step][:a.clips]:\n                for nn in (n, n[:-4] + \".txt\"):\n                    open(os.path.join(data, nn), \"wb\").write(z.read(nn))\n            toml = z.read(\"dataset.toml\").decode(\"utf-8\")\n        open(os.path.join(data, \"dataset.toml\"), \"w\", encoding=\"utf-8\").write(toml.replace(\"DATA_ROOT\", data))\n    toml = os.path.join(data, \"dataset.toml\")\n    nclip = len(glob.glob(os.path.join(vid, \"*.mp4\")))\n    say(f\"자료 {nclip}조각\")\n    src = os.path.join(mt, \"src\", \"musubi_tuner\")\n    env0 = dict(os.environ, CUDA_VISIBLE_DEVICES=\"0\", PYTHONUNBUFFERED=\"1\")\n    res = {\"clips\": nclip, \"gpu\": torch.cuda.get_device_name(0)}\n    rc, sec = run([sys.executable, os.path.join(src, \"wan_cache_latents.py\"), \"--dataset_config\", toml, \"--vae\", os.path.join(m, \"wan_2.1_vae.safetensors\"),\n                   \"--vae_cache_cpu\"], mt, os.path.join(logdir, \"cache_latents.log\"), env0)\n    res[\"cache_latents\"] = dict(rc=rc, sec=round(sec), sec_per_clip=round(sec / max(1, nclip), 1), est_482_min=round(sec / max(1, nclip) * 482 / 60))\n    say(\"잠재 캐시\", res[\"cache_latents\"])\n    rc, sec = run([sys.executable, os.path.join(src, \"wan_cache_text_encoder_outputs.py\"), \"--dataset_config\", toml,\n                   \"--t5\", os.path.join(m, \"models_t5_umt5-xxl-enc-bf16.pth\"), \"--batch_size\", \"8\", \"--fp8_t5\"], mt, os.path.join(logdir, \"cache_text.log\"), env0)\n    if rc:  # T4에서 fp8 저장이 안 되면 bf16 저장으로\n        rc, sec = run([sys.executable, os.path.join(src, \"wan_cache_text_encoder_outputs.py\"), \"--dataset_config\", toml,\n                       \"--t5\", os.path.join(m, \"models_t5_umt5-xxl-enc-bf16.pth\"), \"--batch_size\", \"8\"], mt, os.path.join(logdir, \"cache_text.log\"), env0)\n    res[\"cache_text\"] = dict(rc=rc, sec=round(sec))\n    say(\"글 캐시\", res[\"cache_text\"])\n\n    # 두 방식 동시 학습\n    mw = MemWatch()\n    mw.start()\n    variants = {\"fp16\": (\"dit_fp16\", \"fp16\", 0), \"bf16\": (\"dit_bf16\", \"bf16\", 1 if ng > 1 else 0)}\n    out = {}\n\n    def train(name):\n        dk, mp, gpu = variants[name]\n        o = os.path.join(W, \"out_\" + name)\n        log = os.path.join(logdir, f\"train_{name}.log\")\n        cmd = [sys.executable, \"-m\", \"accelerate.commands.launch\", \"--num_cpu_threads_per_process\", \"1\", \"--num_processes\", \"1\", \"--mixed_precision\", mp,\n               os.path.join(src, \"wan_train_network.py\"), \"--task\", \"t2v-1.3B\", \"--dit\", os.path.join(m, os.path.basename(ev.MODELS[dk][0])),\n               \"--dataset_config\", toml, \"--sdpa\", \"--mixed_precision\", mp, \"--optimizer_type\", \"AdamW\", \"--learning_rate\", \"2e-4\",\n               \"--gradient_checkpointing\", \"--max_data_loader_n_workers\", \"2\", \"--persistent_data_loader_workers\",\n               \"--network_module\", \"networks.lora_wan\", \"--network_dim\", \"32\", \"--network_alpha\", \"16\",\n               \"--timestep_sampling\", \"shift\", \"--discrete_flow_shift\", \"3.0\", \"--max_train_steps\", str(a.steps),\n               \"--seed\", \"42\", \"--output_dir\", o, \"--output_name\", \"test_\" + name]\n        rc, sec = run(cmd, mt, log, dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED=\"1\"), timeout=50 * 60)\n        steps, nan = parse_steps(log)\n        last = max(steps) if steps else 0\n        out[name] = dict(rc=rc, min=round(sec / 60, 1), gpu=gpu, last_step=last, nan=nan,\n                         s_per_it=steps[last][0] if last else None,\n                         loss={k: steps[k][1] for k in sorted(steps) if k in PC_REF or k == last},\n                         est_3000_steps_h=round(steps[last][0] * 3000 / 3600, 1) if last else None)\n        say(f\"학습 {name}:\", out[name])\n\n    names = [\"fp16\", \"bf16\"]\n    if ng > 1:\n        ths = [threading.Thread(target=train, args=(n,)) for n in names]\n        for t in ths:\n            t.start()\n        for t in ths:\n            t.join()\n    else:\n        for n in names:\n            train(n)\n    mw.stop = True\n    res[\"train\"] = out\n    res[\"peak_mem_mib\"] = mw.peak\n    res[\"pc_ref\"] = dict(s_per_it=2.85, loss=PC_REF)\n    json.dump(res, open(os.path.join(logdir, \"summary.json\"), \"w\", encoding=\"utf-8\"), ensure_ascii=False, indent=1)\n    say(\"요약:\", json.dumps(res, ensure_ascii=False))\n    zp = os.path.join(a.out, \"wan_train_test.zip\")\n    with zipfile.ZipFile(zp, \"w\", zipfile.ZIP_DEFLATED) as z:\n        for f in glob.glob(os.path.join(logdir, \"*\")):\n            z.write(f, os.path.relpath(f, a.out))\n    if shutil.which(\"rclone\") and os.path.exists(os.path.expanduser(\"~/.config/rclone/rclone.conf\")):\n        r = subprocess.run([\"rclone\", \"copy\", zp, \"gdrive:AI-sfa/02_분석자료/WanLoRA_평가/\"], capture_output=True, text=True)\n        say(\"Drive 업로드:\", \"성공\" if r.returncode == 0 else \"실패 — Output 탭에서 받아 주세요\")\n    say(f\"완료 → {zp} (요약 summary.json을 Claude에게)\")\n\n\nif __name__ == \"__main__\":\n    main()\n")
!python -u /tmp/kaggle_wan_train_test_worker.py --out /kaggle/working --steps 100